In [1]:
import sys
sys.path.append("..")   # lets the notebook import from src/

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.clean import loadData, cleanData
from src.features import addFeatures
from src.evaluate import evaluateModel

df = loadData("../data/raw/creditcard.csv")
df = cleanData(df)
df = addFeatures(df)

V_COLUMNS = [f"V{i}" for i in range(1, 29)]
SCALE_COLUMNS = ["Amount_log","Time","Hour"]
FEATURE_COLUMNS = V_COLUMNS+SCALE_COLUMNS


def preprocessor(scale_cols):

    col_transformer = ColumnTransformer(
        transformers=[("scale", StandardScaler(), scale_cols)],
        remainder="passthrough"
    )
    return col_transformer


def runExperiment():
    X, y = df[FEATURE_COLUMNS], df["Class"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    models = {
        "logistic_regression": Pipeline([
            ("preprocess", preprocessor(SCALE_COLUMNS)),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ]),
        "random_forest": Pipeline([
            ("preprocess", preprocessor(SCALE_COLUMNS)),
            ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample",
                                           n_jobs=-1, random_state=42)),
        ]),
    }
    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        results[name] = evaluateModel(model, X_test, y_test)

    return pd.DataFrame(results).T


print(runExperiment())

                     precision    recall        f1    pr_auc   roc_auc  \
logistic_regression   0.054967  0.873684  0.103427  0.683011  0.965679   
random_forest         0.985294  0.705263  0.822086  0.806752  0.919334   

                          tn      fp    fn    tp  
logistic_regression  55224.0  1427.0  12.0  83.0  
random_forest        56650.0     1.0  28.0  67.0  
